# Practice Session 04: Basket analysis

Author: <font color="#2680EB">Bo-Yu Chen</font>

E-mail: <font color="#2680EB">8011chen@gmail.com</font>

AI used: <font color="#2680EB">Claude sonnet 5.5 to help me double check. Chatgpt to help me fix the f string error in the notebook.</font>

Date: <font color="#2680EB">19/10/2026</font>

In [1]:
import numpy as np  
import matplotlib.pyplot as plt  
import pandas as pd  
import csv
import gzip

import apyori
from apyori import apriori

# 1. Playing with apyori

In the cell below, my python gave an f string error, I used AI to fix it. So I have changed a bit of it.

In [ ]:
#AI (edited) for f string error
# LEAVE AS-IS

# Pretty-print apyori output
def print_apyori_output(association_rules):
    output = interpret_apyori_output(association_rules)

    for (itemset, rules) in output.items():
        print(f"Rules involving itemset {itemset}")
        for rule in rules:
            print(f"{rule['antecedent']} => {rule['consequent']}")
            print(f"  (sup:{rule['support']:.4f}, conf:{rule['confidence']:.2f}, lift:{rule['lift']:.2f})")
    
        print()

# Returns a dictionary
#  The key is an itemset
#  The value is a list of rules involving that itemset
# Each rule is a dictionary, having
# antecedent, consequent, support, confidence, lift
def interpret_apyori_output(association_results):
    output = {}
    
    for relation_record in association_results:
        itemset = list(relation_record.items)
        
        # Consider only itemsets of two elements
        if len(itemset) > 1: 
        
            output[tuple(itemset)] = []
            support = relation_record.support

            for rules in relation_record.ordered_statistics:
                antecedent = list(rules.items_base)
                consequent = list(rules.items_add)
                           
                confidence = rules.confidence
                lift = rules.lift

                output[tuple(itemset)].append({
                    "antecedent": antecedent,
                    "consequent": consequent,
                    "support": support,
                    "confidence": confidence,
                    "lift": lift,
                })
    return output

<font size="+1" color="#2680EB">**Provide example transactions**. Write at least 20 transactions.</font>

In [ ]:

transactions = [
    ["calculus1", "calculus2"],
    ["python", "machine learning", "calculus1"],
    ["probability", "security", "python"],
    ["assembly", "parallel processing", "machine learning", "python"],
    ["game development", "calculus1", "calculus2", "calculus3"],
    ["probability", "discrete math", "calculus1", "calculus2"],
    ["system programming", "operating system", "virtual machine"],
    ["calculus1"],
    ["calculus4", "calculus3"],
    ["web programming", "python"],
    ["python", "artificial intelligence", "generative AI", "machine learning"],
    ["probability", "random process"],
    ["network", "python"],
    ["system admin", "calculus1", "calculus2", "linear algebra"],
    ["rl", "machine learning", "generative AI", "python"],
    ["artificial intelligence", "rl"],
    ["rl", "machine learning", "calculus1", "calculus2", "python"],
    ["web programming"],
    ["python", "generative AI"],
    ["probability"]
    ] # YOUR CODE HERE

<font size="+1" color="#2680EB">**Run apriori over your transactions**. Execute the apriori algorithm, in which you should obtain at least **two** rules of the form ['A', 'B'] => ['C'], i.e., at least two rules having a 2-itemset in the antecedent and a 1-itemset in the consequent. Modify the transactions until you obtain such rules.</font>

In [4]:
results = list(apriori(transactions,
                       min_support=0.1,
                       min_confidence=0.9,
                       min_lift=1.0))
print_apyori_output(results)

Rules involving itemset ('calculus2', 'calculus1')
['calculus2'] => ['calculus1']
  (sup:0.2500, conf:1.00, lift:2.86)

Rules involving itemset ('generative AI', 'python')
['generative AI'] => ['python']
  (sup:0.1500, conf:1.00, lift:2.22)

Rules involving itemset ('python', 'machine learning')
['machine learning'] => ['python']
  (sup:0.2500, conf:1.00, lift:2.22)

Rules involving itemset ('python', 'calculus1', 'machine learning')
['calculus1', 'machine learning'] => ['python']
  (sup:0.1000, conf:1.00, lift:2.22)
['python', 'calculus1'] => ['machine learning']
  (sup:0.1000, conf:1.00, lift:4.00)

Rules involving itemset ('generative AI', 'python', 'machine learning')
['generative AI', 'machine learning'] => ['python']
  (sup:0.1000, conf:1.00, lift:2.22)

Rules involving itemset ('rl', 'python', 'machine learning')
['rl', 'machine learning'] => ['python']
  (sup:0.1000, conf:1.00, lift:2.22)
['rl', 'python'] => ['machine learning']
  (sup:0.1000, conf:1.00, lift:4.00)



<font size="+1" color="#2680EB">**Answer the questions below**.</font>

**Responses.**

Describe briefly the set of transactions that you created, and how you created it:

* This is a list of all the courses that each CS student might take. I generated based on that each student might have some interest, so they are more likely to take some kind of courses.

Which are the two rules that you obtained?

* ['python', 'calculus1'] => ['machine learning'] and ['generative AI', 'machine learning'] => ['python']

For one of those rules, can you provide the calculation of *support*?

* For ['python', 'calculus1'] => ['machine learning'], the support of 'python' and 'calculus1' and 'machine learning' is calculated by:
total transactions = 20
number of transactions containing 'python' and 'calculus1' and 'machine learning' = 2
so Support('python' and 'calculus1' and 'machine learning') = 2/20 = 0.1

For the same rule, can you provide the calculation of *confidence*?

* For ['python', 'calculus1'] => ['machine learning'], the confidence is calculated by:
total transactions = 20
number of transactions containing 'python' and 'calculus1' and 'machine learning' = 2
number of transactions containing 'python' and 'calculus1' = 2
so Confidence(['python', 'calculus1'] => ['machine learning']) = 2/2 = 1


# 2. Load grocery shopping dataset

In [6]:
# LEAVE AS-IS

# File name
INPUT_TRANSACTIONS = "groceries.csv.gz"

# Load
MAX_TRANSACTIONS = 2000
transactions = []
with gzip.open(INPUT_TRANSACTIONS, "rt") as inputfile:
    reader = csv.reader(inputfile, delimiter=",")
    
    for item_list in reader:
        transactions.append(item_list)
        
        if len(transactions) >= MAX_TRANSACTIONS:
            break
        
               
print(f"Read {len(transactions)} transactions")

Read 2000 transactions


# 3. Extract association rules and comment on them

<font size="+1" color="#2680EB">**Run apriori over these transactions**.</font>

In [37]:
num_transactions = len(transactions)

# YOUR CODE HERE

min_support_absolute = 40
min_support=float(min_support_absolute)/num_transactions

results = list(apriori(
    transactions,
    min_support=min_support,
    min_confidence=0.4,
    min_lift=1.2
    ))

print_apyori_output(results)

Rules involving itemset ('whole milk', 'brown bread')
['brown bread'] => ['whole milk']
  (sup:0.0255, conf:0.42, lift:1.64)

Rules involving itemset ('butter', 'whole milk')
['butter'] => ['whole milk']
  (sup:0.0285, conf:0.47, lift:1.82)

Rules involving itemset ('curd', 'whole milk')
['curd'] => ['whole milk']
  (sup:0.0320, conf:0.48, lift:1.88)

Rules involving itemset ('whole milk', 'domestic eggs')
['domestic eggs'] => ['whole milk']
  (sup:0.0295, conf:0.46, lift:1.80)

Rules involving itemset ('rolls/buns', 'frankfurter')
['frankfurter'] => ['rolls/buns']
  (sup:0.0255, conf:0.40, lift:1.96)

Rules involving itemset ('whole milk', 'frozen vegetables')
['frozen vegetables'] => ['whole milk']
  (sup:0.0245, conf:0.50, lift:1.93)

Rules involving itemset ('margarine', 'whole milk')
['margarine'] => ['whole milk']
  (sup:0.0265, conf:0.43, lift:1.67)

Rules involving itemset ('whole milk', 'napkins')
['napkins'] => ['whole milk']
  (sup:0.0205, conf:0.41, lift:1.57)

Rules involv

<font size="+1" color="#2680EB">**Comment on the transactions found**.</font>

**Responses.**

Select one association rule that seems to describe something *causal*. Hint: consider someone buying ingredients for a recipe.

* Rules involving itemset ('butter', 'whole milk')
['butter'] => ['whole milk']
  (sup:0.0285, conf:0.47, lift:1.82)

buying butter will likely to buy whole milk, as these are two basic ingredients to make pastries, so people who bought butter will likely buy whole milk

Select one association rule that seems to be obtained merely because people tend to do trips to the supermarket to purchase several perishable, fast-expiration products.

* Rules involving itemset ('whole milk', 'pork')
['pork'] => ['whole milk']
  (sup:0.0245, conf:0.44, lift:1.69)
pork and milk are both things that have a shorter expiration date, while they are both foods, they does not have a very strong relationship in daily life, people likely bought them together just because they want to buy these things in one trip.

# 4. Re-implement computation of confidence and lift

In [38]:
# LEAVE AS-IS

# Use apyori to obtain the number of transactions containing a list of itemsets
def compute_support_count(transactions, itemsets):
    num_transactions = len(transactions)
    
    # The instructor used AI to understand how to use apyori internals (Gemini 3.6 Flash)
    manager = apyori.TransactionManager.create(transactions)
    supports = []
    
    for itemset in itemsets:
        supports.append(int(num_transactions * manager.calc_support(frozenset(itemset))))
    return supports

def compute_support(transactions, itemsets):
    counts = compute_support_count(transactions, itemsets)
    return [count / len(transactions) for count in counts]

# Test this
print(compute_support_count(transactions, [["whole milk"], ["soda", "waffles"]]))
print(compute_support(transactions, [["whole milk"], ["soda", "waffles"]]))

[517, 24]
[0.2585, 0.012]


<font size="+1" color="#2680EB">**Write code that computes support, confidence, and lift**. Check programmatically using assert that each of these is within 0.001 of the value returned by Apyori.</font>

In the cell below, my interpreter gave following error, I used AI to fix it, so I have changed a bit of the code.

  Cell In[39], line 7
    print(f"{rule["antecedent"]} => {rule["consequent"]}")
                   ^
SyntaxError: f-string: unmatched '['

This is using the variable outside of the funciton transaction.

This is still using the help of the library to calculate the support.

In [ ]:
#AI (edited) for f string error

epsilon = 0.001

def print_apyori_output_checking(association_rules):
    output = interpret_apyori_output(association_rules)

    for (itemset, rules) in output.items():
        print(f"Rules involving itemset {itemset}")
        for rule in rules:
            print(f"{rule['antecedent']} => {rule['consequent']}")
            print(f"  apyori (sup:{rule['support']:.4f}, conf:{rule['confidence']:.2f}, lift:{rule['lift']:.2f})")

            # YOUR CODE HERE
            support_A, support_B, support_A_B = compute_support(transactions, [rule['antecedent'], rule['consequent'], rule['antecedent'] + rule['consequent']])
            support = support_A_B
            confidence = support_A_B / support_A
            lift = confidence / support_B
            
            # assertions
            assert(abs(support - rule['support']) < epsilon)
            assert(abs(confidence - rule['confidence']) < epsilon)
            assert(abs(lift - rule['lift']) < epsilon)
            
            print(f"  me (sup:{support:.4f}, conf:{confidence:.2f}, lift:{lift:.2f})")
    
        print()
             
print_apyori_output_checking(results)

Rules involving itemset ('whole milk', 'brown bread')
['brown bread'] => ['whole milk']
  apyori (sup:0.0255, conf:0.42, lift:1.64)
  me (sup:0.0255, conf:0.42, lift:1.64)

Rules involving itemset ('butter', 'whole milk')
['butter'] => ['whole milk']
  apyori (sup:0.0285, conf:0.47, lift:1.82)
  me (sup:0.0285, conf:0.47, lift:1.82)

Rules involving itemset ('curd', 'whole milk')
['curd'] => ['whole milk']
  apyori (sup:0.0320, conf:0.48, lift:1.88)
  me (sup:0.0320, conf:0.48, lift:1.88)

Rules involving itemset ('whole milk', 'domestic eggs')
['domestic eggs'] => ['whole milk']
  apyori (sup:0.0295, conf:0.46, lift:1.80)
  me (sup:0.0295, conf:0.46, lift:1.80)

Rules involving itemset ('rolls/buns', 'frankfurter')
['frankfurter'] => ['rolls/buns']
  apyori (sup:0.0255, conf:0.40, lift:1.96)
  me (sup:0.0255, conf:0.40, lift:1.96)

Rules involving itemset ('whole milk', 'frozen vegetables')
['frozen vegetables'] => ['whole milk']
  apyori (sup:0.0245, conf:0.50, lift:1.93)
  me (sup:0

<font size="+2" color="#00A365">I hereby declare that I completed this practice myself, that I have not copied it from others, or from external sources without citing them, and that I have followed the AI instructions stated above.</font>